# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTD504/flyrank-ai-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal Checks & Empirical Verdicts

1. **Signal 1: Content Staleness (`freshness_tier` / `days_since_last_update >= 180`)**
   - **Verdict:** **OPPOSITE**
   - **Reasoning:** Stale articles (`181+` days) exhibit a decay rate of only **7.47%**, significantly below the global baseline of **15.16%** and lower than fresh articles (`14.10%`). This is caused by a baseline floor effect: stale content has already lost most of its organic traffic, leaving little click volume to drop further. Pure content age alone is a misleading signal for decay velocity.

2. **Signal 2: Impression Trajectory (`impressions_last_30d < impressions_prev_30d`)**
   - **Verdict:** **CONFIRMED**
   - **Reasoning:** Articles with stable or growing impressions have a **0.00%** decay rate ($0 / 10,284$ rows). Articles with dropping impressions jump to a **23.07%** decay rate ($4,548 / 19,716$ rows). Impression drop is a mandatory filtering condition for organic traffic decay.

---

### Plain Words Rule Definition
A page is queued for urgent refresh if its impression volume is dropping, weighted by the absolute scale of impression loss and a moderate penalty for content older than 90 days.

* **Score Formula:**
  $$
  \text{score} = \max(0, \text{impressions_prev_30d} - \text{impressions_last_30d}) \times (1.0 + 0.5 \times \mathbb{I}(\text{days_since_last_update} \ge 90))
  $$

* **Reason Codes:**
  - `high_volume_drop_and_stale`: Impression drop $\ge 100$ and age $\ge 90$ days.
  - `high_volume_drop_fresh_content`: Impression drop $\ge 100$ and age $< 90$ days.
  - `low_volume_drop_stale`: Impression drop $< 100$ and age $\ge 90$ days.
  - `low_volume_drop_fresh`: Impression drop $< 100$ and age $< 90$ days.
  - `stable_or_growing_impressions`: No impression loss recorded ($\text{score} = 0$).

* **Action Labels:**
  - `URGENT_REFRESH_PRIORITY_1`: Baseline Score $\ge 100$.
  - `MONITOR_REFRESH_PRIORITY_2`: Baseline Score $> 0$ and $< 100$.
  - `NO_ACTION_STABLE`: Baseline Score $= 0$.

In [1]:
!git clone https://github.com/PTD504/flyrank-ai-ml-internship.git

Cloning into 'flyrank-ai-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 49), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 8.17 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# 1. Load Starter Dataset
data_dir = "flyrank-ai-ml-internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_dir)

# 2. Define Ground Truth Target (from w02 Data Contract / Framing)
df['target_is_decaying'] = ((df['trend_direction'] == 'down') & (df['clicks_last_30d'] < df['clicks_prev_30d'])).astype(int)
base_rate = df['target_is_decaying'].mean()

print(f"=== GROUND TRUTH BASE RATE ===")
print(f"Total rows (N): {len(df):,}")
print(f"Base Decay Rate (target_is_decaying = 1): {base_rate:.4f} ({base_rate * 100:.2f}%)\n")

# ---------------------------------------------------------
# SIGNAL CHECK 1: Content Staleness (FlyRank Flag Signal)
# Claim: Older / untended content (stale) decays at a higher rate.
# ---------------------------------------------------------
df['is_stale_180d'] = (df['days_since_last_update'] >= 180).astype(int)

signal1_table = df.groupby('freshness_tier').agg(
    n=('target_is_decaying', 'count'),
    decay_count=('target_is_decaying', 'sum'),
    decay_rate=('target_is_decaying', 'mean')
).reset_index()

signal1_table['decay_rate_pct'] = (signal1_table['decay_rate'] * 100).round(2)

print("=== SIGNAL CHECK 1: Content Freshness Tier (Staleness) ===")
print(signal1_table[['freshness_tier', 'n', 'decay_count', 'decay_rate_pct']].to_string(index=False))
print("\n" + "-"*60 + "\n")

# ---------------------------------------------------------
# SIGNAL CHECK 2: Impression Decay Ratio (Volume Velocity)
# Claim: Drop in impressions (impressions_last_30d < impressions_prev_30d) indicates loss of search visibility.
# ---------------------------------------------------------
df['impressions_dropping'] = (df['impressions_last_30d'] < df['impressions_prev_30d']).astype(int)

signal2_table = df.groupby('impressions_dropping').agg(
    n=('target_is_decaying', 'count'),
    decay_count=('target_is_decaying', 'sum'),
    decay_rate=('target_is_decaying', 'mean')
).reset_index()

signal2_table['decay_rate_pct'] = (signal2_table['decay_rate'] * 100).round(2)
signal2_table['impression_status'] = signal2_table['impressions_dropping'].map({0: 'Stable/Growing Impressions', 1: 'Dropping Impressions'})

print("=== SIGNAL CHECK 2: Impression Trajectory ===")
print(signal2_table[['impression_status', 'n', 'decay_count', 'decay_rate_pct']].to_string(index=False))

=== GROUND TRUTH BASE RATE ===
Total rows (N): 30,000
Base Decay Rate (target_is_decaying = 1): 0.1516 (15.16%)

=== SIGNAL CHECK 1: Content Freshness Tier (Staleness) ===
freshness_tier     n  decay_count  decay_rate_pct
          0-30 20480         2887           14.10
          181+   174           13            7.47
         31-90   175           16            9.14
        91-180  9171         1632           17.80

------------------------------------------------------------

=== SIGNAL CHECK 2: Impression Trajectory ===
         impression_status     n  decay_count  decay_rate_pct
Stable/Growing Impressions 10284            0            0.00
      Dropping Impressions 19716         4548           23.07


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import pandas as pd
import numpy as np

# Create output directory for artifacts
os.makedirs("work/outputs", exist_ok=True)

# 1. Load Starter Dataset
data_dir = "flyrank-ai-ml-internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_dir)

# 2. Re-construct Ground Truth Target
df['target_is_decaying'] = ((df['trend_direction'] == 'down') & (df['clicks_last_30d'] < df['clicks_prev_30d'])).astype(int)

# 3. Calculate Rule Components
df['impression_loss'] = np.maximum(0, df['impressions_prev_30d'] - df['impressions_last_30d'])
df['is_stale_90d'] = (df['days_since_last_update'] >= 90).astype(int)

# 4. Compute Transparent Baseline Score
df['baseline_score'] = df['impression_loss'] * (1.0 + 0.5 * df['is_stale_90d'])

# 5. Assign Reason Codes
def assign_reason_code(row):
    if row['impression_loss'] <= 0:
        return "stable_or_growing_impressions"
    elif row['is_stale_90d'] == 1 and row['impression_loss'] >= 100:
        return "high_volume_drop_and_stale"
    elif row['is_stale_90d'] == 0 and row['impression_loss'] >= 100:
        return "high_volume_drop_fresh_content"
    elif row['is_stale_90d'] == 1:
        return "low_volume_drop_stale"
    else:
        return "low_volume_drop_fresh"

df['reason_code'] = df.apply(assign_reason_code, axis=1)

# 6. Assign Action Labels
def assign_action_label(row):
    if row['baseline_score'] >= 100:
        return "URGENT_REFRESH_PRIORITY_1"
    elif row['baseline_score'] > 0:
        return "MONITOR_REFRESH_PRIORITY_2"
    else:
        return "NO_ACTION_STABLE"

df['action_label'] = df.apply(assign_action_label, axis=1)

# 7. Sort and Rank Order
ranked_df = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)
ranked_df['rank'] = ranked_df.index + 1

# 8. Compute Precision@K Metrics
base_rate = df['target_is_decaying'].mean()

def precision_at_k(df_sorted, k_count):
    return df_sorted.head(k_count)['target_is_decaying'].mean()

top_50_precision = precision_at_k(ranked_df, 50)
top_20pct_k = int(len(ranked_df) * 0.20)
top_20pct_precision = precision_at_k(ranked_df, top_20pct_k)

print(f"=== BASELINE RULE EVALUATION METRICS ===")
print(f"Base Decay Rate (Random Floor):          {base_rate:.4f} ({base_rate*100:.2f}%)")
print(f"Baseline Rule Precision@50:               {top_50_precision:.4f} ({top_50_precision*100:.2f}%)")
print(f"Baseline Rule Precision@Top 20% (K={top_20pct_k}): {top_20pct_precision:.4f} ({top_20pct_precision*100:.2f}%)\n")

# 9. Write Ranked Queue CSV
output_cols = [
    'rank', 'content_id', 'client_id', 'baseline_score', 'reason_code',
    'action_label', 'impressions_prev_30d', 'impressions_last_30d',
    'days_since_last_update', 'clicks_prev_30d', 'clicks_last_30d', 'target_is_decaying'
]
output_path = "work/outputs/baseline_action_score.csv"
ranked_df[output_cols].to_csv(output_path, index=False)

print(f"✅ Successfully wrote ranked queue to: {output_path}")
print("\n=== TOP 10 RANKED QUEUE SAMPLE ===")
print(ranked_df[output_cols].head(10).to_string(index=False))

=== BASELINE RULE EVALUATION METRICS ===
Base Decay Rate (Random Floor):          0.1516 (15.16%)
Baseline Rule Precision@50:               0.7800 (78.00%)
Baseline Rule Precision@Top 20% (K=6000): 0.4577 (45.77%)

✅ Successfully wrote ranked queue to: work/outputs/baseline_action_score.csv

=== TOP 10 RANKED QUEUE SAMPLE ===
 rank           content_id         client_id  baseline_score                    reason_code              action_label  impressions_prev_30d  impressions_last_30d  days_since_last_update  clicks_prev_30d  clicks_last_30d  target_is_decaying
    1 content_5fe46e04994d client_4e07408562        146992.5     high_volume_drop_and_stale URGENT_REFRESH_PRIORITY_1                218786                120791                     104              250              220                   1
    2 content_9532f197bbc8 client_4e07408562         97377.0     high_volume_drop_and_stale URGENT_REFRESH_PRIORITY_1                174235                109317                     104       

### Detailed Top-20 Audit

Below is the qualitative evaluation of the top 20 prioritized articles generated by our rule baseline:

1. **Rank 1 (`content_5fe46e04994d`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). **What would make it wrong:** High impression drop ($218k \to 120k$) paired with click drop ($250 \to 220$). Would be wrong if search intent shifted permanently away from this topic, making refreshes ineffective.
2. **Rank 2 (`content_9532f197bbc8`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($174k \to 109k$) and click drop ($1440 \to 1176$). Would be wrong if traffic loss is strictly macroeconomic or seasonal.
3. **Rank 3 (`content_2c2606c5d176`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($164k \to 104k$) and click drop ($859 \to 578$). Would be wrong if lost impressions were low-intent vanity queries with no conversion value.
4. **Rank 4 (`content_cb112fce36be`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($124k \to 72k$) and click drop ($171 \to 134$). Would be wrong if competitor domain authority simply outranks this page regardless of content updates.
5. **Rank 5 (`content_c8e9d6ab9013`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** **WEAK / FALSE POSITIVE (`target_is_decaying = 0`)**. Impressions dropped ($111k \to 63k$), but clicks remained flat at **0**. Would be wrong (and IS wrong) because spending editorial resources on a page that generates zero clicks yields zero ROI.
6. **Rank 6 (`content_8c19996aa890`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** High (`target_is_decaying = 1`). Fresh content ($20$ days) experiencing steep drop ($161k \to 89k$ imps, $389 \to 251$ clicks). Would be wrong if search engines are still in the index re-ranking phase (Google dance).
7. **Rank 7 (`content_66b4046cc144`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** High (`target_is_decaying = 1`). Massive impression loss ($69k \to 7.4k$) and click drop ($15 \to 4$). Would be wrong if page suffered a technical indexing bug rather than a content freshness issue.
8. **Rank 8 (`content_3d94572c3a35`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression loss ($78k \to 37k$) and click loss ($185 \to 103$). Would be wrong if lost queries were seasonal.
9. **Rank 9 (`content_ec66c58d9826`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** High (`target_is_decaying = 1`). Severe traffic loss ($57k \to 2.7k$ imps, $381 \to 16$ clicks). Would be wrong if the page URL structure changed without proper 301 redirects.
10. **Rank 10 (`content_89fcb6f35525`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($79k \to 43k$) and click drop ($333 \to 253$). Would be wrong if loss is limited to non-converting queries.
11. **Rank 11 (`content_8b36799b7e44`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($53k \to 19k$), clicks $8 \to 7$. Would be wrong if base click volume ($8$) is too low to justify high priority.
12. **Rank 12 (`content_11fcfd65d94c`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** High (`target_is_decaying = 1`). Impression loss ($76k \to 44k$) and click loss ($106 \to 52$). Would be wrong if position loss was driven by SERP layout changes (AI Overviews).
13. **Rank 13 (`content_813e88069237`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** **WEAK / FALSE POSITIVE (`target_is_decaying = 0`)**. Impressions dropped ($94k \to 62k$), but clicks actually **INCREASED** ($47 \to 56$). Would be wrong because CTR improved, offsetting impression loss.
14. **Rank 14 (`content_124763d39ca5`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** **WEAK / FALSE POSITIVE (`target_is_decaying = 0`)**. Impressions dropped ($43k \to 11k$), but clicks increased ($5 \to 6$). Would be wrong due to low absolute click base and CTR compensation.
15. **Rank 15 (`content_d07ea098353c`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** **WEAK / FALSE POSITIVE (`target_is_decaying = 0`)**. Impressions dropped ($43k \to 13k$), but clicks grew ($7 \to 9$). Would be wrong because actual organic traffic expanded despite search snippet impression contraction.
16. **Rank 16 (`content_05e9b4cd9ccf`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** **WEAK / FALSE POSITIVE (`target_is_decaying = 0`)**. Impression drop ($69k \to 40k$), but clicks grew ($52 \to 65$). Would be wrong because user traffic expanded (+25%).
17. **Rank 17 (`content_73e5410650b9`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($50k \to 8k$) and click drop ($22 \to 10$). Would be wrong if cannibalized by a newer internal article.
18. **Rank 18 (`content_8d3971bfd976`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($54k \to 12k$) and click drop ($13 \to 10$). Would be wrong if overall search volume for the intent contracted globally.
19. **Rank 19 (`content_4c36c775b818`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_fresh_content` | **Confidence:** High (`target_is_decaying = 1`). Impression drop ($125k \to 83k$) and click drop ($662 \to 548$). Would be wrong if ranking loss is temporary volatility.
20. **Rank 20 (`content_54baba704595`)**: `URGENT_REFRESH_PRIORITY_1` | Reason: `high_volume_drop_and_stale` | **Confidence:** **WEAK / FALSE POSITIVE (`target_is_decaying = 0`)**. Impression loss ($50k \to 22k$), but clicks remained stable at $1 \to 1$. Would be wrong because click volume was negligible to begin with.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load generated baseline queue
df_queue = pd.read_csv("work/outputs/baseline_action_score.csv")

# Extract Top 20 rows
top_20 = df_queue.head(20).copy()

print("=== TOP 20 DETAILED AUDIT TABLE ===")
audit_cols = [
    'rank', 'content_id', 'baseline_score', 'reason_code',
    'impressions_prev_30d', 'impressions_last_30d',
    'clicks_prev_30d', 'clicks_last_30d',
    'days_since_last_update', 'target_is_decaying'
]
print(top_20[audit_cols].to_string(index=False))

# Calculate False Positives in Top 20
top_20_fp = top_20[top_20['target_is_decaying'] == 0]
print("\n" + "="*60)
print(f"=== WEAK PICKS AUDIT (False Positives in Top 20: {len(top_20_fp)} / 20) ===")
if len(top_20_fp) > 0:
    print(top_20_fp[['rank', 'content_id', 'baseline_score', 'clicks_prev_30d', 'clicks_last_30d', 'target_is_decaying']].to_string(index=False))
else:
    print("No False Positives found in Top 20.")

# Feature Leakage Safeguard Check
forbidden_cols = ['trend_direction', 'trend_pct', 'health_score', 'needs_ctr_fix', 'is_quick_win', 'is_declining_label']
leaked = [col for col in forbidden_cols if col in df_queue.columns]

print("\n=== FEATURE LEAKAGE SAFEGUARD CHECK ===")
if not leaked:
    print("CONFIRMED: Zero future-window or product-flag columns present in output CSV.")
else:
    print(f"WARNING: Potential leakage columns detected in queue output: {leaked}")

=== TOP 20 DETAILED AUDIT TABLE ===
 rank           content_id  baseline_score                    reason_code  impressions_prev_30d  impressions_last_30d  clicks_prev_30d  clicks_last_30d  days_since_last_update  target_is_decaying
    1 content_5fe46e04994d        146992.5     high_volume_drop_and_stale                218786                120791              250              220                     104                   1
    2 content_9532f197bbc8         97377.0     high_volume_drop_and_stale                174235                109317             1440             1176                     104                   1
    3 content_2c2606c5d176         89746.5     high_volume_drop_and_stale                164079                104248              859              578                     104                   1
    4 content_cb112fce36be         78048.0     high_volume_drop_and_stale                124500                 72468              171              134                     104     

## 4. Weak picks + leakage check

### Weak Picks Analysis & Structural Rule Flaws

Our hand-written baseline rule achieved a **78.00% Precision@50**, but auditing the Top 20 reveals **6 False Positives (30% error rate in Top 20)**:

1. **Zero-Click Blindspot (Rank 5):** `content_c8e9d6ab9013` lost 48,559 impressions, scoring a massive **72,838.5**, placing it in the Top 5. However, it generated **0 clicks in both periods**. The rule failed because it evaluates impression loss in isolation without checking baseline traffic viability.
2. **The CTR Compensation Trap (Ranks 13, 14, 15, 16):** These articles suffered significant impression drops, yet their **clicks actually increased** (e.g., Rank 16 grew from 52 to 65 clicks). This occurs when ranking position or title snippet optimization improves CTR despite lower query impression broadness. A linear rule based solely on impressions falsely flags growing pages as "decaying."
3. **Low-Baseline Noise (Rank 20):** `content_54baba704595` lost 27,425 impressions but maintained 1 click. High impression volatility on low-click pages inflates baseline scores without real business impact.

---

### Feature Leakage Safeguard Verification

- **Future-Window Columns:** `trend_direction` and `trend_pct` were strictly excluded from score calculation.
- **Product Decision Flags:** `health_score`, `needs_ctr_fix`, and `is_quick_win` were excluded to prevent circular learning.
- **Confirmation:** Verified programmatically via explicit column audits — the exported queue CSV (`baseline_action_score.csv`) contains zero prohibited outcome/flag signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.